# Birefringence Analysis

Optics TP6 — L3 Physics, Sorbonne Université

Two experiments:
- Stress-induced birefringence (photoelastic effect)
- Spectral fringes (cannelures) to measure dn*e from fringe positions

Transmitted intensity through a birefringent plate between crossed polarizers:
$$I(\lambda, \theta) = I_0 \sin^2(2\theta) \sin^2\left(\frac{\pi \Delta n \cdot e}{\lambda}\right)$$

Minima when $\Delta n \cdot e / \lambda = m$ (integer).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import sys
sys.path.append('..')

from optics import birefringence

---
## Part 1: Stress birefringence

Plexiglass sample under stress — we count fringes as a function of displacement.

In [ ]:
x_data = np.array([0, 2.1, 4.2, 6.2, 8.3, 10.5, 12.4]) * 1e-2  # cm to m
y_data = np.array([0, 1, 2, 3, 4, 5, 6])  # fringe orders
x_erreurs = np.full_like(x_data, 0.5e-3)   # +/-0.5 mm

print(f"{len(x_data)} points, displacement up to {x_data.max()*100:.1f} cm")

In [ ]:
# linear fit: m = a*x + b
def linear(x, a, b):
    return a * x + b

popt, pcov = curve_fit(linear, x_data, y_data, p0=[1e4, 0])
a_best, b_best = popt
u_a = np.sqrt(pcov[0][0])
u_b = np.sqrt(pcov[1][1])

pred = linear(x_data, a_best, b_best)
r2 = 1 - np.sum((y_data - pred)**2) / np.sum((y_data - np.mean(y_data))**2)

print(f"Slope: a = {a_best:.2f} +/- {u_a:.2f} fringes/m")
print(f"Intercept: b = {b_best:.4f} +/- {u_b:.4f}")
print(f"R2 = {r2:.6f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6),
                               gridspec_kw={'height_ratios': [3, 1]},
                               sharex=True)

ax1.errorbar(x_data*100, y_data, xerr=x_erreurs*100,
             fmt='o', capsize=4, label='data')
x_fit = np.linspace(x_data.min(), x_data.max(), 100)
ax1.plot(x_fit*100, linear(x_fit, a_best, b_best), 'r-', lw=2, label='linear fit')
ax1.set_ylabel('Fringe order m')
ax1.set_title('Stress birefringence: fringes vs displacement')
ax1.legend()
ax1.grid(alpha=0.3)

residuals = y_data - pred
ax2.plot(x_data*100, residuals, 'o')
ax2.axhline(0, color='red', ls='--', lw=1)
ax2.set_xlabel('Displacement (cm)')
ax2.set_ylabel('Residual')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# very linear -- each 2 cm adds one fringe

Linear relationship confirms the photoelastic effect: fringe count is proportional to applied stress.

---
## Part 2: Spectral fringes (cannelures)

White light through a birefringent plate between crossed polarizers. We measure the wavelengths where transmission minima occur.

In [ ]:
# wavelengths of spectral minima (nm)
lambda_min_nm = np.array([565.8, 572.9, 580.1, 587.6, 595.2, 603.0, 611.3, 619.6, 628.1, 637.1])
m_data = np.arange(len(lambda_min_nm))  # orders 0-9

lambda_min = lambda_min_nm * 1e-9  # to m
inv_lambda = 1.0 / lambda_min

print(f"{len(lambda_min_nm)} minima from {lambda_min_nm.min():.1f} to {lambda_min_nm.max():.1f} nm")
print(f"Average spacing: ~{np.mean(np.diff(lambda_min_nm)):.1f} nm")

### Fit: 1/lambda vs m

From $\Delta n \cdot e = m\lambda$ we get $1/\lambda = m / (\Delta n \cdot e)$, so plotting 1/lambda vs m should be linear.

In [ ]:
def linear(m, A, B):
    return A * m + B

popt, pcov = curve_fit(linear, m_data, inv_lambda)
A_best, B_best = popt
u_A = np.sqrt(pcov[0][0])

pred_spec = linear(m_data, A_best, B_best)
r2_spec = 1 - np.sum((inv_lambda - pred_spec)**2) / np.sum((inv_lambda - np.mean(inv_lambda))**2)

print(f"Slope A = {A_best:.2e} +/- {u_A:.2e} m^-1")
print(f"R2 = {r2_spec:.6f}")

# dn*e from slope
dn_e = 1.0 / A_best
u_dn_e = u_A / A_best**2
print(f"\ndn*e = {dn_e*1e6:.1f} +/- {u_dn_e*1e6:.1f} um")

# rough thickness estimate for quartz
e_est = dn_e / 0.009
print(f"If quartz (dn ~ 0.009): e ~ {e_est*1e3:.2f} mm")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# left: minima wavelengths
axes[0].plot(m_data, lambda_min_nm, 'o-', ms=7)
axes[0].set_xlabel('Fringe order m')
axes[0].set_ylabel('lambda (nm)')
axes[0].set_title('Minima wavelengths')
axes[0].grid(alpha=0.3)

# right: 1/lambda vs m
axes[1].plot(m_data, inv_lambda*1e-6, 'o', ms=7, label='data')
m_fit = np.linspace(m_data.min(), m_data.max(), 100)
axes[1].plot(m_fit, linear(m_fit, A_best, B_best)*1e-6, 'r-', lw=2, label='fit')
axes[1].set_xlabel('Fringe order m')
axes[1].set_ylabel('1/lambda (x10^6 m^-1)')
axes[1].set_title('Linear analysis')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# residuals
res_spec = (inv_lambda - pred_spec) * 1e-6

plt.figure(figsize=(8, 3))
plt.plot(m_data, res_spec, 'o')
plt.axhline(0, color='red', ls='--', lw=1)
plt.xlabel('Fringe order m')
plt.ylabel('Residual (x10^6 m^-1)')
plt.title('Residuals')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# residuals are small, no clear pattern

## Conclusion

**Stress birefringence**: fringe count scales linearly with displacement (R2 ~ 1), confirming the photoelastic effect.

**Spectral fringes**: 1/$\lambda$ vs m is very linear, which gives us dn*e directly from the slope. The estimated thickness is in the right ballpark for a quartz plate, though we're assuming dn ~ 0.009 which is a rough number.

One thing I notice is that the minima are roughly equally spaced in lambda over this range, but the proper linear relationship is in 1/$\lambda$ — the difference is subtle since we're looking at a narrow wavelength window.

---
*Optical Analysis Toolkit — data from L3 Physics TP6, Sorbonne Université*